In [2]:
import numpy as np

from Historia.shared.design_utils import read_labels
from Historia.shared.design_utils import lhd
from SIMULATION_library import simulator_utils

In [24]:
folder_experiment_name = "HCM/10GH00962/scenarios/6"
basefolder             = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
datafolder             = f"{basefolder}/data"

fields = ['ToRORd',
		  'ToRORd_land',
		  'EP',
		  'mechanics',
		  'circadapt'
		  ]

# fields = ['circadapt']

Read parameter ranges and labels and sample using LHD:

In [25]:
N = 1000 # To play safe


for f in fields:
    I_ = np.loadtxt(f"{datafolder}/I_{f}.txt",dtype=float)

    xlabels_ = read_labels(f"{datafolder}/xlabels_{f}.txt")

    # check if dimensions match
    if len(xlabels_)!=I_.shape[0] and len(xlabels_) > 1: 
        print(f"Read {datafolder}/I_{f}.txt")
        print("xlabels: ")
        print(xlabels_)
        print("Intervals: ")
        print(I_)
        raise Exception(f"xlabels and I sizes for field {f} do not match, xlabels is {len(xlabels_)} and intervals are {I_.shape[0]}")

    X_ = lhd(np.atleast_2d(I_),N)
    np.savetxt(f"{datafolder}/X_{f}.txt",X_,fmt="%g")

Convert to json files for later:

In [6]:
simulator_utils.X_to_json(labels_fields = fields,
                          datafolder    = datafolder,
                          outputfolder  = f"{basefolder}/json_files",
                          default_json  = f"{basefolder}/json_files/default.json")

generating json file...
EP
(1000,)
circadapt
(1000, 2)
mechanics
(1000, 3)
ToRORd
(1000, 2)
ToRORd_land
(1000,)


mkdir: cannot create directory ‘/media/croderog/SeagateExpansionDrive/HCM/10GH00962/scenarios/6/json_files’: File exists


Merge all the datasets for ARCHER

In [27]:
X_array = []
xlabels_array = []

for field in fields:
    X_ = np.loadtxt(f"{datafolder}/X_{field}.txt", dtype=float)

    # Check if X_ has only one column, reshape to 2D array
    if X_.ndim == 1:
        X_ = X_.reshape(-1, 1)

    X_array.append(X_)

    xlabels_ = read_labels(f"{datafolder}/xlabels_{field}.txt")
    xlabels_array.append(xlabels_)

# X = np.concatenate(X_array, axis=1)
    
X = np.hstack(X_array)
xlabels = np.concatenate(xlabels_array, axis=0)

np.savetxt(f"{datafolder}/X.txt",X,fmt="%g")
np.savetxt(f"{datafolder}/xlabels.txt",xlabels,fmt="%s")